# Supplier Compliance Intelligence — Sensitivity Output Audit

This notebook validates the existing Phase 1 sensitivity-analysis exports without recalculating the scenarios or writing files. The scenario method is implemented in [`run_analysis.py`](../run_analysis.py); this notebook checks the generated CSVs and summarizes supplier movement.

The source workbook and the malformed legacy notebook are left untouched. Run this notebook with the project `.venv` kernel from the project directory or a child directory.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 20)

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'reports' / 'supplier_scenario_comparison.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate reports/supplier_scenario_comparison.csv from the current directory.')

REPORTS = PROJECT_ROOT / 'reports'
scenario = pd.read_csv(REPORTS / 'supplier_scenario_comparison.csv')
category_totals = pd.read_csv(REPORTS / 'scenario_category_comparison.csv').set_index('Category')
priority = pd.read_csv(REPORTS / 'supplier_priority_tiers.csv')

print(f'Project root: {PROJECT_ROOT.name}')
print(f'Scenario comparison: {scenario.shape[0]} suppliers')
print(f'Priority tiers: {priority.shape[0]} suppliers')

Project root: supplier-compliance-intelligence
Scenario comparison: 800 suppliers
Priority tiers: 800 suppliers


In [2]:
checks = []

def check(description, passed):
    checks.append((description, bool(passed)))
    print(f"{'PASS' if passed else 'FAIL'}: {description}")

check('Scenario export contains 800 unique suppliers', len(scenario) == 800 and scenario['Supplier_ID'].nunique() == 800)
check('Priority-tier export contains the same 800 suppliers', len(priority) == 800 and set(priority['Supplier_ID']) == set(scenario['Supplier_ID']))

scenarios = ['Original', 'Criticality_Emphasis', 'Info_Gap_Emphasis']
for name in scenarios:
    score_col = f'Risk_Score_{name}'
    category_col = f'Risk_Category_{name}'
    rank_col = f'Rank_{name}'
    check(f'{name} scores, categories, and ranks have no missing values',
          scenario[[score_col, category_col, rank_col]].notna().all().all())
    actual = scenario[category_col].value_counts()
    matches = all(int(actual.get(category, 0)) == int(category_totals.loc[category, name])
                  for category in ['High', 'Medium', 'Low'])
    check(f'{name} category totals match the category export', matches)

check('Original scenario score matches the source risk score within 0.01',
      (scenario['Risk_Score_Original'] - scenario['Overall_Risk_Score']).abs().max() <= 0.01)
check('All scenario ranks are between 1 and 800',
      scenario[[f'Rank_{name}' for name in scenarios]].ge(1).all().all()
      and scenario[[f'Rank_{name}' for name in scenarios]].le(800).all().all())

failed = [description for description, passed in checks if not passed]
print(f'\n{len(checks) - len(failed)} / {len(checks)} checks passed')
if failed:
    raise AssertionError(f'Failed checks: {failed}')

PASS: Scenario export contains 800 unique suppliers
PASS: Priority-tier export contains the same 800 suppliers
PASS: Original scores, categories, and ranks have no missing values
PASS: Original category totals match the category export
PASS: Criticality_Emphasis scores, categories, and ranks have no missing values
PASS: Criticality_Emphasis category totals match the category export
PASS: Info_Gap_Emphasis scores, categories, and ranks have no missing values
PASS: Info_Gap_Emphasis category totals match the category export
PASS: Original scenario score matches the source risk score within 0.01
PASS: All scenario ranks are between 1 and 800

10 / 10 checks passed


## Scenario profile and top-181 movement

The top-181 set uses the original High-supplier count as a fixed capacity. Negative rank change means a supplier moves upward toward rank 1.

In [3]:
category_profile = pd.DataFrame({
    name: scenario[f'Risk_Category_{name}'].value_counts().reindex(['High', 'Medium', 'Low'], fill_value=0)
    for name in scenarios
})
display(category_profile)

top_sets = {
    name: set(scenario.loc[scenario[f'Rank_{name}'] <= 181, 'Supplier_ID'])
    for name in scenarios
}
top181_rows = []
for name in ['Criticality_Emphasis', 'Info_Gap_Emphasis']:
    top181_rows.append({
        'Scenario': name,
        'Top 181 count': len(top_sets[name]),
        'Retained': len(top_sets['Original'] & top_sets[name]),
        'Entrants': len(top_sets[name] - top_sets['Original']),
        'Exits': len(top_sets['Original'] - top_sets[name]),
    })
top181_summary = pd.DataFrame(top181_rows)
display(top181_summary)

,Original,Criticality_Emphasis,Info_Gap_Emphasis
High,181,148,151
Medium,366,371,381
Low,253,281,268


,Scenario,Top 181 count,Retained,Entrants,Exits
0,Criticality_Emphasis,181,164,17,17
1,Info_Gap_Emphasis,181,165,16,16


## Largest individual movements

Context columns are taken directly from the supplier-level export. These are prioritisation movements under alternative weights, not changes in underlying supplier conditions.

In [4]:
movement_rows = []
for name, change_col in [
    ('Criticality_Emphasis', 'Rank_Change_Criticality_Emphasis'),
    ('Info_Gap_Emphasis', 'Rank_Change_Info_Gap_Emphasis'),
]:
    for direction, row in [('Largest upward', scenario.loc[scenario[change_col].idxmin()]),
                           ('Largest downward', scenario.loc[scenario[change_col].idxmax()])]:
        movement_rows.append({
            'Scenario': name,
            'Movement': direction,
            'Supplier_ID': row['Supplier_ID'],
            'Original rank': int(row['Rank_Original']),
            'Scenario rank': int(row[f'Rank_{name}']),
            'Rank change': int(row[change_col]),
            'Criticality score': row['Criticality_Score'],
            'Overdue severity score': row['Overdue_Severity_Score'],
        })
display(pd.DataFrame(movement_rows))

,Scenario,Movement,Supplier_ID,Original rank,Scenario rank,Rank change,Criticality score,Overdue severity score
0,Criticality_Emphasis,Largest upward,SUP-0699,437,217,-220,100,0.00
1,Criticality_Emphasis,Largest downward,SUP-0118,524,708,184,20,25.00
2,Info_Gap_Emphasis,Largest upward,SUP-0210,411,281,-130,20,75.83
3,Info_Gap_Emphasis,Largest downward,SUP-0788,458,567,109,100,0.00


## Category transitions

Rows show the original category and columns show the category under each alternative weighting.

In [5]:
for name in ['Criticality_Emphasis', 'Info_Gap_Emphasis']:
    print(f'\n{name} vs Original')
    transitions = pd.crosstab(
        scenario['Risk_Category_Original'],
        scenario[f'Risk_Category_{name}'],
    ).reindex(index=['High', 'Medium', 'Low'], columns=['High', 'Medium', 'Low'], fill_value=0)
    display(transitions)


Criticality_Emphasis vs Original


Risk_Category_Criticality_Emphasis,High,Medium,Low
Risk_Category_Original,,,
High,136,45,0
Medium,12,311,43
Low,0,15,238



Info_Gap_Emphasis vs Original


Risk_Category_Info_Gap_Emphasis,High,Medium,Low
Risk_Category_Original,,,
High,147,34,0
Medium,4,338,24
Low,0,9,244
